In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import re
import os
from openai import OpenAI
from dotenv import load_dotenv
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings('ignore')
load_dotenv()

# ============================================================================
# Project paths
# ============================================================================
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODEL_DIR = RESULTS_DIR / 'model'
TABLE_DIR = RESULTS_DIR / 'table'

print("Project paths ready")

print("=" * 80)
print("Agent #3: Ensemble Quality Assurance (6 Specialists, CF Alignment = Verify-then-Judge)")
print("=" * 80)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
MODEL = os.getenv('LLM_MODEL', 'gpt-4o-mini')

# ============================================================================
# 1. Feature Name Mapping
# ============================================================================
BASE_TERM_MAPPING = {
    'FN1_1': 'Current Assets', 'FN1_2': 'Non-Current Assets', 'FN1_3': 'Quick Assets',
    'FN1_4': 'Inventory', 'FN1_5': 'Tangible Assets', 'FN1_6': 'Work in Process',
    'FN1_7': 'Cash', 'FN1_8': 'Cash Equivalents', 'FN1_9': 'Marketable Securities',
    'FN1_10': 'Cash and Cash Equivalents', 'FN1_11': 'Accounts Receivable',
    'FN1_11_2': 'Loss on Disposal of Receivables', 'FN1_11_3': 'Intangible Assets',
    'FN1_11_4': 'Investment Assets', 'FN1_14': 'Current Liabilities',
    'FN1_15': 'Short-Term Borrowings', 'FN1_16': 'Borrowings', 'FN1_17': 'Accounts Payable',
    'FN1_18': 'Non-Current Liabilities', 'FN1_19': 'Total Liabilities',
    'FN1_20': 'Paid-in Capital', 'FN1_21': 'Capital Surplus', 'FN1_21_1': 'Paid-in Capital (Detail)',
    'FN1_22': 'Retained Earnings', 'FN1_22_1': 'Capital Adjustments',
    'FN1_22_2': 'Accumulated Other Comprehensive Income', 'FN1_23': 'Reserves',
    'FN1_24': 'Total Equity', 'FN3_10_1': 'Liquidation Value', 'FN3_11': 'Net Working Capital',
    'FN3_11_1': 'Net Borrowings',
    'FN2_2': 'Cost of Goods Sold', 'FN2_2_1': 'Gross Profit', 'FN2_3': 'SG&A Expenses',
    'FN2_3_1': 'Pre-Tax Income', 'FN2_3_2': 'Prior-Year Pre-Tax Income', 'FN2_3_3': 'Corporate Tax',
    'FN2_3_4': 'Income from Continuing Operations', 'FN2_3_5': 'Discontinued Operations Gain/Loss',
    'FN2_4': 'Financial Expenses', 'FN2_5': 'Operating Income', 'FN2_5_1': 'Prior-Year Operating Income',
    'FN2_7': 'Non-Operating Income', 'FN2_8': 'Non-Operating Expenses', 'FN2_9': 'Pre-Tax Net Income',
    'FN2_10': 'Net Income', 'FN3_1': 'Cash Flow', 'FN3_2': 'Operating Cash Flow',
    'FN3_2_1': 'Investing Cash Flow', 'FN3_2_2': 'Financing Cash Flow', 'FN3_4_1': 'Interest Expense',
    'FN3_4_2': 'Bond Interest', 'FN3_7': 'EBIT', 'FN3_8': 'EBITDA',
    'FN3_3': 'Debt Service Coverage Ratio', 'FN3_6': 'Reserve Ratio', 'FN3_10': 'Liquidation Value Ratio',
    'asset_growth_rate': 'Total Asset Growth Rate', 'revenue_growth_rate': 'Revenue Growth Rate',
    'operating_income_growth': 'Operating Income Growth Rate', 'net_income_growth': 'Net Income Growth Rate',
    'equity_growth_rate': 'Equity Growth Rate',
}

def get_readable_name(feature_name):
    if feature_name in BASE_TERM_MAPPING:
        return BASE_TERM_MAPPING[feature_name]
    if feature_name.endswith('_to_assets'):
        base = feature_name[:-len('_to_assets')]
        base_term = BASE_TERM_MAPPING.get(base, base)
        return f"{base_term} (% of Total Assets)"
    if feature_name.endswith('_to_revenue'):
        base = feature_name[:-len('_to_revenue')]
        base_term = BASE_TERM_MAPPING.get(base, base)
        return f"{base_term} (% of Revenue)"
    return feature_name

# ============================================================================
# 2. Non-CF-Alignment Specialist Prompts
# ============================================================================
SPECIALIST_PROMPTS = {
    "logic_flow": """
    You are a [Logical Structure Analyst].
    Disregard numerical accuracy entirely and evaluate only whether
    the paragraph flow and causal reasoning are logically sound.
    Verify whether 'A therefore B' arguments are valid.
    """,
    "actionability": """
    You are a [Field Implementation Consultant].
    Evaluate whether this proposal is executable starting tomorrow,
    or whether it is abstract and impractical.
    Check whether specific responsible parties, deadlines, and methodologies are stated.
    """,
    "business_insight": """
    You are a [Business Strategist].
    Assess whether this is a report that merely lists numbers,
    or whether it contains genuine strategic insight for the firm's survival.
    Verify whether it appropriately reflects the characteristics of the firm's sector.
    """,
    "completeness": """
    You are a [Compliance Officer].
    Evaluate whether the report includes all 6 required sections
    and whether each section has sufficient substance.
    Focus on structural and formal compliance.
    """,
    "clarity": """
    You are a [Communication Specialist].
    Evaluate whether the report is written in clear, accessible language
    suitable for executive readers.
    Check for excessive jargon or vague expressions.
    """
}

WEIGHTS = {
    "cf_alignment": 0.25,
    "actionability": 0.25,
    "business_insight": 0.20,
    "logic_flow": 0.15,
    "completeness": 0.10,
    "clarity": 0.05
}

# ============================================================================
# 3. CF ALIGNMENT — Verify-then-Judge
# ============================================================================

# ---- FIXED: build_cf_reference now uses the SAME top-N-by-magnitude
# selection as Agent #2's _format_financial_guide, so ground truth is built
# only from values the report was actually given (and instructed) to cite.
# Validated on a 10-firm sample: mean cf_alignment score rose from 1.36
# (unbounded reference, unfairly penalizing reports for omitting variables
# never shown to them) to 2.80 (top-10-matched reference). ----
def build_cf_reference(agent1_row, feature_list, min_change=0.001, top_n=10):
    candidates = []
    for feat in feature_list:
        try:
            orig = float(agent1_row[f'Original_{feat}'])
            target = float(agent1_row[f'CF_{feat}'])
            change = target - orig
            if abs(change) > min_change:
                pct_change = abs(change / abs(orig)) if orig != 0 else abs(change)
                readable = get_readable_name(feat)
                candidates.append((pct_change, readable, orig, target))
        except (KeyError, ValueError):
            continue

    candidates.sort(key=lambda x: x[0], reverse=True)

    reference = {}
    for _, readable, orig, target in candidates[:top_n]:
        reference[readable] = (orig, target)

    return reference

# ---- Stage A: Extractor (with defensive schema normalization) ----
EXTRACTOR_PROMPT = """
You are a [Numeric Extractor]. Your only job is to extract numbers.

Extract every numeric value that appears in the report body (percentages,
ratios, decimals, currency figures). For each number, note the short phrase
of context immediately surrounding it (max 15 words).

Do NOT judge whether the numbers are correct. Do NOT compare them to
anything. Only extract and contextualize.

Output JSON:
{
    "extracted_numbers": [
        {"value": "the number exactly as written in the text", "context": "short surrounding phrase"}
    ]
}
"""

def stage_a_extract_numbers(report_text):
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": EXTRACTOR_PROMPT},
                {"role": "user", "content": f"[Report Content]\n{report_text}"}
            ],
            response_format={"type": "json_object"},
            temperature=0.0
        )
        raw = json.loads(response.choices[0].message.content).get('extracted_numbers', [])

        normalized = []
        for item in raw:
            if isinstance(item, dict) and 'value' in item:
                normalized.append(item)
            elif isinstance(item, str):
                normalized.append({'value': item, 'context': ''})
            elif isinstance(item, (list, tuple)) and len(item) >= 1:
                normalized.append({'value': item[0], 'context': item[1] if len(item) > 1 else ''})
        return normalized
    except Exception as e:
        return []

# ---- Stage B: Deterministic Matcher (pure Python) ----
def stage_b_deterministic_match(extracted_numbers, cf_reference_values, tol=0.01):
    def parse_number(s):
        cleaned = re.sub(r'[^\d.\-]', '', str(s))
        try:
            return float(cleaned)
        except ValueError:
            return None

    extracted_values = []
    for item in extracted_numbers:
        if isinstance(item, dict) and 'value' in item:
            parsed = parse_number(item['value'])
            if parsed is not None:
                extracted_values.append(parsed)

    ground_truth_values = []
    for orig, target in cf_reference_values.values():
        ground_truth_values.extend([orig, target])

    matched_ground_truth = set()
    for ev in extracted_values:
        for i, gt in enumerate(ground_truth_values):
            if i in matched_ground_truth:
                continue
            if abs(gt) > 1:
                if abs(ev - gt) < max(tol, abs(gt) * 0.01):
                    matched_ground_truth.add(i)
                    break
            else:
                if abs(ev - gt) < tol:
                    matched_ground_truth.add(i)
                    break

    n_ground_truth = len(ground_truth_values)
    n_matched = len(matched_ground_truth)
    match_rate = n_matched / n_ground_truth if n_ground_truth > 0 else 0.0

    return {
        'n_ground_truth_values': n_ground_truth,
        'n_matched': n_matched,
        'match_rate': round(match_rate, 3),
        'unmatched_ground_truth': [
            ground_truth_values[i] for i in range(n_ground_truth) if i not in matched_ground_truth
        ]
    }

# ---- Stage C: Judge (deterministic rubric) ----
def stage_c_judge_alignment(match_result):
    match_rate = match_result['match_rate']
    n_gt = match_result['n_ground_truth_values']

    if match_rate >= 0.95:
        score = 5
        reason = f"{match_result['n_matched']}/{n_gt} ground-truth CF values ({match_rate*100:.0f}%) found verbatim (within tolerance) in the report."
    elif match_rate >= 0.75:
        score = 4
        reason = f"{match_result['n_matched']}/{n_gt} ground-truth CF values ({match_rate*100:.0f}%) matched; minor narrative paraphrase acceptable per rubric."
    elif match_rate >= 0.50:
        score = 3
        reason = f"Only {match_result['n_matched']}/{n_gt} ground-truth CF values ({match_rate*100:.0f}%) matched; at least one key figure appears miscalculated or omitted."
    elif match_rate >= 0.25:
        score = 2
        reason = f"Just {match_result['n_matched']}/{n_gt} ground-truth CF values ({match_rate*100:.0f}%) matched; most figures are missing or incorrect."
    else:
        score = 1
        reason = f"Almost no ground-truth CF values found in the report ({match_result['n_matched']}/{n_gt}, {match_rate*100:.0f}%)."

    return {'score': score, 'reason': reason}

def evaluate_cf_alignment_verify_then_judge(report_text, cf_reference_values):
    extracted = stage_a_extract_numbers(report_text)
    match_result = stage_b_deterministic_match(extracted, cf_reference_values)
    judged = stage_c_judge_alignment(match_result)
    judged['match_rate'] = match_result['match_rate']
    judged['n_matched'] = match_result['n_matched']
    judged['n_ground_truth'] = match_result['n_ground_truth_values']
    return judged

# ============================================================================
# 4. Other Specialist Agent Execution
# ============================================================================
def call_specialist_agent(agent_name, report_text):
    system_persona = SPECIALIST_PROMPTS[agent_name]
    user_prompt = f"""
    Evaluate the following financial consulting report strictly from your
    specialist perspective [{agent_name}].

    [Report Content]
    {report_text}

    [Evaluation Criteria]
    Assign a score on a 1 (worst) to 5 (best) scale
    and provide a one-sentence justification citing specific evidence.

    [Output Format (JSON)]
    {{
        "score": score (integer 1-5),
        "reason": "evaluation justification with specific evidence"
    }}
    """
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_persona},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.2
        )
        return agent_name, json.loads(response.choices[0].message.content)
    except Exception as e:
        return agent_name, {"score": 1, "reason": f"Error: {str(e)}"}

# ============================================================================
# 5. Ensemble Evaluator
# ============================================================================
class EnsembleEvaluator:
    def __init__(self, feature_list):
        self.feature_list = feature_list

    def evaluate_report(self, report_row, agent1_row):
        report_content = report_row['report_content']
        cf_reference_values = build_cf_reference(agent1_row, self.feature_list)

        results = {}

        results['cf_alignment'] = evaluate_cf_alignment_verify_then_judge(
            report_content, cf_reference_values
        )

        with ThreadPoolExecutor(max_workers=5) as executor:
            future_to_agent = {
                executor.submit(call_specialist_agent, name, report_content): name
                for name in SPECIALIST_PROMPTS.keys()
            }
            for future in as_completed(future_to_agent):
                agent_name, result = future.result()
                results[agent_name] = result

        total_score = 0
        details = {}
        for name, weight in WEIGHTS.items():
            score = results[name].get('score', 1)
            total_score += score * weight
            details[f"score_{name}"] = score
            details[f"reason_{name}"] = results[name].get('reason', 'N/A')

        details['cf_alignment_match_rate'] = results['cf_alignment'].get('match_rate')
        details['cf_alignment_n_matched'] = results['cf_alignment'].get('n_matched')
        details['cf_alignment_n_ground_truth'] = results['cf_alignment'].get('n_ground_truth')

        if total_score >= 3.60:
            decision = "Pass"
        elif total_score >= 3.30:
            decision = "Conditional Pass"
        else:
            decision = "Reject"

        return {
            'company_id': report_row['company_id'],
            'SIC_CD_3': report_row.get('SIC_CD_3', None),
            'final_score': round(total_score, 2),
            'decision': decision,
            **details
        }

# ============================================================================
# 6. Execution
# ============================================================================
reports = pd.read_csv(DATA_DIR / 'agent2_consulting_reports_4industry.csv')
agent1_data = pd.read_csv(DATA_DIR / 'agent1_interpretation_results_4industry.csv')

feature_list = [c.replace('Original_', '') for c in agent1_data.columns if c.startswith('Original_')]

merged = reports.merge(
    agent1_data, left_on='company_id', right_on='ID', how='left', suffixes=('', '_agent1')
)

evaluator = EnsembleEvaluator(feature_list)
evaluated_reports = []

print(f"\nStarting 6-specialist evaluation (cf_alignment = verify-then-judge, top-10 matched) for {len(merged)} reports...")

from tqdm import tqdm
for idx, row in tqdm(merged.iterrows(), total=len(merged), desc="Evaluating"):
    eval_result = evaluator.evaluate_report(row, row)
    evaluated_reports.append(eval_result)

df_eval = pd.DataFrame(evaluated_reports)
output_path = DATA_DIR / 'agent3_ensemble_results_4industry.csv'
df_eval.to_csv(output_path, index=False, encoding='utf-8-sig')

print("\n" + "=" * 80)
print("Evaluation complete")
print("=" * 80)

pass_cnt = (df_eval['decision'] == 'Pass').sum()
cond_cnt = (df_eval['decision'] == 'Conditional Pass').sum()
reject_cnt = (df_eval['decision'] == 'Reject').sum()
avg_score = df_eval['final_score'].mean()

print(f"Total evaluated: {len(df_eval)}")
print(f"- Pass:             {pass_cnt} ({pass_cnt/len(df_eval)*100:.1f}%)")
print(f"- Conditional Pass: {cond_cnt} ({cond_cnt/len(df_eval)*100:.1f}%)")
print(f"- Reject:           {reject_cnt} ({reject_cnt/len(df_eval)*100:.1f}%)")
print(f"- Average score:    {avg_score:.2f}")

print(f"\n[cf_alignment score distribution]")
print(df_eval['score_cf_alignment'].value_counts().sort_index())
print(f"Mean cf_alignment score: {df_eval['score_cf_alignment'].mean():.2f}")
print(f"Mean match_rate: {df_eval['cf_alignment_match_rate'].mean():.3f}")

print(f"\n[By industry]")
print(pd.crosstab(df_eval['SIC_CD_3'], df_eval['decision']))

print(f"\n[Mean scores by evaluation dimension]")
score_cols = ['score_cf_alignment', 'score_logic_flow', 'score_actionability',
              'score_business_insight', 'score_completeness', 'score_clarity']
print(df_eval[score_cols].mean().round(2))

if not df_eval.empty:
    sample = df_eval.iloc[0]
    print(f"\n[Sample Evaluation: Company {sample['company_id']} ({sample['SIC_CD_3']})]")
    print(f"Composite Score: {sample['final_score']} ({sample['decision']})")
    print("-" * 40)
    print(f"1. CF Alignment    ({sample['score_cf_alignment']}): {sample['reason_cf_alignment']}")
    print(f"2. Logic & Flow    ({sample['score_logic_flow']}): {sample['reason_logic_flow']}")
    print(f"3. Actionability   ({sample['score_actionability']}): {sample['reason_actionability']}")
    print(f"4. Business Insight({sample['score_business_insight']}): {sample['reason_business_insight']}")
    print(f"5. Completeness    ({sample['score_completeness']}): {sample['reason_completeness']}")
    print(f"6. Clarity         ({sample['score_clarity']}): {sample['reason_clarity']}")

Project paths ready
Agent #3: Ensemble Quality Assurance (6 Specialists, CF Alignment = Verify-then-Judge)

Starting 6-specialist evaluation (cf_alignment = verify-then-judge, top-10 matched) for 542 reports...


Evaluating: 100%|██████████████████████████████████████████████████████████████████| 542/542 [1:48:26<00:00, 12.00s/it]


Evaluation complete
Total evaluated: 542
- Pass:             133 (24.5%)
- Conditional Pass: 280 (51.7%)
- Reject:           129 (23.8%)
- Average score:    3.42

[cf_alignment score distribution]
score_cf_alignment
1      8
2    131
3    260
4    111
5     32
Name: count, dtype: int64
Mean cf_alignment score: 3.05
Mean match_rate: 0.604

[By industry]
decision  Conditional Pass  Pass  Reject
SIC_CD_3                                
F42                     47    22      27
G46                     91    44      53
G47                     71    42      25
L68                     71    25      24

[Mean scores by evaluation dimension]
score_cf_alignment        3.05
score_logic_flow          3.63
score_actionability       2.98
score_business_insight    3.98
score_completeness        4.20
score_clarity             3.13
dtype: float64

[Sample Evaluation: Company 9 (L68)]
Composite Score: 3.05 (Reject)
----------------------------------------
1. CF Alignment    (2): Just 6/20 ground-truth C